# Pandas

In [30]:
import numpy as np
import pandas as pd
import seaborn as sns                      # 통계 전용 시각화 - matplotlib 같이 사용 가능 

import matplotlib.pyplot as plt            # 그래프 출력 옵션 (Detail)
import matplotlib as mpl                   # 외적인(큰틀) 옵션 - 한글, 사이즈 등  

mpl.rc('font', family='Malgun Gothic')     # 폰트 경로 - C:\Windows\Fonts\Malgun Gothic
plt.rcParams['axes.unicode_minus']=False   # 마이너스 설정 

## 결측치
- 데이터에서 값이 없거나 비어 있는 상태

In [8]:
dye_na=pd.read_csv('./Data/염색공정_결측치.csv', encoding='cp949')
dye_na.shape

(50, 7)

In [10]:
dye_na.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   설비번호      49 non-null     str    
 1   투입중량_kg   49 non-null     float64
 2   염색길이_m    49 non-null     float64
 3   지시온도      50 non-null     float64
 4   진행온도      48 non-null     float64
 5   공정진행시간_%  50 non-null     float64
 6   염색색차_DE   50 non-null     float64
dtypes: float64(6), str(1)
memory usage: 3.1 KB


In [13]:
dye_na.isna().sum()

설비번호        1
투입중량_kg     1
염색길이_m      1
지시온도        0
진행온도        2
공정진행시간_%    0
염색색차_DE     0
dtype: int64

### 결측치 삭제

In [16]:
na=dye_na.copy()

In [18]:
na.dropna(inplace=True)

In [20]:
na.isna().sum()

설비번호        0
투입중량_kg     0
염색길이_m      0
지시온도        0
진행온도        0
공정진행시간_%    0
염색색차_DE     0
dtype: int64

In [22]:
na.reset_index(drop=True, inplace=True)      # 인덱스 리셋

- 결측 삭제 기준
    - (결측치 행 수 / 전체 행수) * 100 : 5 ~ 10% 삭제
    - KAMP(중소벤쳐기업부 제조 AI)의 품질 관리 메뉴얼 30%

In [26]:
total=dye_na.isnull().any(axis=1)     # 열 방향, 즉 행 단위로 확인 / 하나라도 있으면 true
total.sum() / len(dye_na) * 100

10.0

### 결측치 대체
- 연속형(숫자)
    - 평균 또는 중앙값으로 대체
    - 시계열 데이터 : 앞/뒤 값으로 채우기
- 범주형(문자) : 최빈수 대체

In [47]:
edit=dye_na.copy()

In [49]:
edit.isna().sum()

설비번호        1
투입중량_kg     1
염색길이_m      1
지시온도        0
진행온도        2
공정진행시간_%    0
염색색차_DE     0
dtype: int64

In [51]:
edit.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
투입중량_kg,49.0,138.87,89.09,0.00,63.60,128.00,189.10,350.8
염색길이_m,49.0,213.47,276.17,0.00,33.00,96.00,334.00,1600.0
지시온도,50.0,81.39,33.73,30.00,59.18,82.25,109.85,130.0
진행온도,48.0,81.16,32.63,21.50,54.90,82.20,106.58,130.0
공정진행시간_%,50.0,42.01,30.05,0.17,18.20,36.24,66.94,99.2
염색색차_DE,50.0,2.06,3.00,0.20,0.45,1.16,1.86,14.4


In [37]:
def sub(series):
    # 기초 통계 계산
    q1=series.quantile(0.25)              #  63.6  하한 값0.51
    q3=series.quantile(0.75)              #  189.1 상한 값1.87
    iqr=q3 - q1                           #  125.5  
    # print(q1, q3, iqr)

    # 하한값, 상한값 계산
    lower_bound=q1 - (1.5 * iqr)            # -124.65
    upper_bound=q3 + (1.5 * iqr)            #  0.8499
    # print(lower_bound, upper_bound)
    # print(series.min(), series.max())     # 0.0, 350.8

    # 이상치 여부
    is_outlier=False
    if series.min() < lower_bound:      # 기준선 보다 작으면 이상치   0.0 > -124.65
        print(f"하위 이상치 존재! (최솟값 {series.min() } < 하한값 {lower_bound:.2f})")
        is_outlier=True

    elif series.max() > upper_bound:   # 기준선 보다 크면 이상치    350.8 > 0.8499
        print(f"상위 이상치 존재! (최댓값 {series.max() } > 상한값 {upper_bound:.2f})")
        is_outlier=True

    # 최종 결정
    if is_outlier:
        print("중앙값 사용 권장")
    else:
        print("평균값 사용 권장")

In [60]:
# 투입중량 : 평균 값
sub(edit['투입중량_kg'])

평균값 사용 권장


In [55]:
weight=edit['투입중량_kg'].mean()
weight

edit['투입중량_kg']=edit['투입중량_kg'].fillna(weight)
edit.isna().sum()

설비번호        1
투입중량_kg     0
염색길이_m      1
지시온도        0
진행온도        2
공정진행시간_%    0
염색색차_DE     0
dtype: int64

In [63]:
# 염새길이 : 중위 값
sub(edit['염색길이_m'])

상위 이상치 존재! (최댓값 1600.0 > 상한값 785.50)
중앙값 사용 권장


In [66]:
length=edit['염색길이_m'].median()
edit['염색길이_m']=edit['염색길이_m'].fillna(length)
edit.isna().sum()

설비번호        1
투입중량_kg     0
염색길이_m      0
지시온도        0
진행온도        2
공정진행시간_%    0
염색색차_DE     0
dtype: int64

In [76]:
# 진행 온도 : 평균 값, 중앙 값
sub(edit['진행온도'])

평균값 사용 권장


In [78]:
# 진행온도 >> 중앙값
# temp=edit['지시온도'].mean()
# edit['지시온도']=edit['지시온도'].fillna(temp)

# EX) 앞(forword), 뒤(backward)
edit['진행온도']=edit['진행온도'].ffill()      # bfill()
edit.isna().sum()

설비번호        1
투입중량_kg     0
염색길이_m      0
지시온도        0
진행온도        0
공정진행시간_%    0
염색색차_DE     0
dtype: int64

- 범주형(문자) : 빈도수
    - 설비번호 
        - 설비별 성능 비교가 분석의 목적이면 삭제
        - 교차 확인 : 해당 데이터의 작업 시간이나, 작업자 정보가 있다면, 로그데이터 확인해서 찾아보기
        - 삭제하기엔 아까운 데이터라면 설비번호 모르는 그룹으로 묶어서 관리

In [442]:
# edit['설비번호']=edit['설비번호'].fillna('Unknown')
# edit['설비번호']=edit['설비번호'].fillna('999')

In [82]:
# 최빈값으로 대체 경우 
# 예] 작업자명, 작업시간(공정시간 10분, 30분, 60분)
mode=edit['설비번호'].mode()[0]
mode

'FCM08'

In [87]:
edit['설비번호']=edit['설비번호'].fillna(mode)
edit.isna().sum()

설비번호        0
투입중량_kg     0
염색길이_m      0
지시온도        0
진행온도        0
공정진행시간_%    0
염색색차_DE     0
dtype: int64

## 이상치
* 정상 범위에서 크게 벗어난 값
* 이상치가 들어 있으면 분석 결과에서 왜곡이 발생한다.
* 이상치 >> 결측치 : 삭제 또는 대체

In [89]:
edit.describe().T

,count,mean,std,min,25%,50%,75%,max
투입중량_kg,50.0,138.867347,88.174679,0.000000,64.200000,133.433673,188.025000,350.80000
염색길이_m,49.0,213.469388,276.170471,0.000000,33.000000,96.000000,334.000000,1600.00000
지시온도,50.0,81.386000,33.728314,30.000000,59.175000,82.250000,109.850000,130.00000
진행온도,48.0,81.160417,32.630359,21.500000,54.900000,82.200000,106.575000,130.00000
공정진행시간_%,50.0,42.011600,30.047728,0.170000,18.202500,36.240000,66.940000,99.20000
염색색차_DE,50.0,2.058314,3.002984,0.200785,0.453264,1.164509,1.860052,14.40188


In [92]:
cond=(edit['투입중량_kg'] == 0) | (edit['염색길이_m'] == 0 )
df=edit[cond]
df.shape
df.head()

,설비번호,투입중량_kg,염색길이_m,지시온도,진행온도,공정진행시간_%,염색색차_DE
29,FCM07,341.8,0.0,30.0,33.3,1.79,3.928609
32,FCM11,0.0,600.0,78.1,77.1,34.73,3.033640


In [95]:
# 이상치 >> 결측치 
edit=edit.replace(0, np.nan)

In [98]:
edit.isna().sum()

설비번호        0
투입중량_kg     1
염색길이_m      1
지시온도        0
진행온도        0
공정진행시간_%    0
염색색차_DE     0
dtype: int64

In [105]:
edit.dropna(inplace=True)

In [107]:
edit.isna().sum()

설비번호        0
투입중량_kg     0
염색길이_m      0
지시온도        0
진행온도        0
공정진행시간_%    0
염색색차_DE     0
dtype: int64

In [109]:
edit.reset_index(drop=True, inplace=True)      # 인덱스 리셋

## 그룹화 & 집계함수
- 예] df.groupby('company')['age'].mean()
    - 'company' 열의 값에 따라 그룹화
    - 그룹화된 각 그룹에서 'age' 열
    - 선택된 'age' 열의 각 그룹별 평균을 계산

In [34]:
dye=pd.read_csv('./Data_Set//염색공정_수정.csv', encoding='cp949')
dye.head()

,설비번호,투입중량_kg,염색길이_m,투입중량/길이,지시온도,진행온도,공정진행시간_%,염색색차_DE,불량여부
0,FCM08,44.6,41.0,1.087805,30.0,10.9,0.19,0.753096,0
1,FCM08,44.6,41.0,1.087805,30.0,10.3,0.39,0.753096,0
2,FCM08,44.6,41.0,1.087805,30.0,10.3,0.58,0.753096,0
3,FCM08,44.6,41.0,1.087805,30.0,10.2,0.77,0.753096,0
4,FCM08,44.6,41.0,1.087805,30.0,10.8,0.97,0.753096,0


In [20]:
mach_no=dye.groupby('설비번호')                   # 설비번호별 그룹화

- 그룹 함수 종류 :

In [23]:
# mach_no['염색색차_DE'].describe().round(2)      # 설비번호별 염색색차_DE 기술 통계
mach_no['염색색차_DE'].mean().round(2)            # 설비번호별 염색색차_DE 평균

설비번호
FCM05    2.81
FCM06    1.18
FCM07    2.71
FCM08    1.89
FCM11    1.56
FCM12    3.95
FCM15    0.40
Name: 염색색차_DE, dtype: float64

In [25]:
mach_no['염색색차_DE'].agg(['mean', 'count'])     # 설비번호별 염색색차_DE 두개 이상 그룹

,mean,count
설비번호,,
FCM05,2.811437,6171
FCM06,1.180983,5774
FCM07,2.705375,1667
FCM08,1.886812,8952
FCM11,1.561275,4442
FCM12,3.953758,3699
FCM15,0.395969,1903


In [27]:
result=mach_no['염색색차_DE'].agg(['mean', 'count'])
result.columns=['평균색차', '측정 수']                 # 컬럼명 변경 가능
result

,평균색차,측정 수
설비번호,,
FCM05,2.811437,6171
FCM06,1.180983,5774
FCM07,2.705375,1667
FCM08,1.886812,8952
FCM11,1.561275,4442
FCM12,3.953758,3699
FCM15,0.395969,1903


## 불량률 원인분석
- 공정
    - 온도 : 지시온도, 진행온도
    - 원단 : 중량, 길이, 투입중량/길이
    - 공정진행시간_%
- 설비 : 설비번호
- 원인분석시: 조건별 비교, 구간분석, 상관분석

In [36]:
(dye['불량여부'].mean() * 100).round(2)   # 전체 불량률

15.19

### 공정 
- 지시온도, 진행온도, 투입중량/길이, 공정진행시간_%

In [41]:
defect=dye.groupby("불량여부")

#### 온도
- 조건별 비교 : 그룹별 평균, 합계, 비율

In [44]:
# 1) 평균값으로 원인 분석 
# 온도:1.5, 투입중량/길이: 확연한 차이 (얇은 원단 또는 원단의 밀도 낮음)

defect[['지시온도','진행온도', '투입중량/길이', '공정진행시간_%']].mean().round(2)

,지시온도,진행온도,투입중량/길이,공정진행시간_%
불량여부,,,,
0,85.24,85.04,5.51,49.72
1,83.75,83.48,1.50,50.18


In [46]:
# 2) 온도의 이상치 확인
# 0인 냉각 상태에서는 불량이 발생하지 않는다. (염색 특성상 온도 가열 필요)

defect[['지시온도', '진행온도']].describe().round(2)

지시온도                                                  진행온도         \
        count   mean    std  min   25%   50%    75%    max    count   mean   
불량여부                                                                         
0     27655.0  85.24  31.40  0.0  70.0  83.8  113.0  135.0  27655.0  85.04   
1      4953.0  83.75  31.34  5.4  70.0  81.3  111.6  135.0   4953.0  83.48   

                                             
        std   min   25%   50%    75%    max  
불량여부                                         
0     31.28   0.0  67.2  83.8  112.6  137.4  
1     31.33  16.5  66.8  81.5  111.2  137.3

- 구간 분석 : 온도 구간별 불량률
    -  구간너비 = (최대값 - 최솟값) / bins
        -  27.48 = (137.4 - 0.0) / 5
    -  시작점 결정 : 최솟값인 0.0에서 전체 범위의 0.1%(0.1374)
        -  0.0 - 0.1374 = -0.1374
    - 1구간 : -0.1374 + 27.48 = 27.3426
    - 2구간 : 27.34 + 27.48 = 54.96
    - 3구간 : 54.96 + 27.48 = 82.44
    - 4구간 : 82.44 + 27.48 = 109.92
    - 5구간 : 09.92 + 27.48 = 137.4

In [51]:
# 3) 진행온도 구간 분석
temp_cut=pd.cut(dye['진행온도'], bins=5)
# temp_cut

dye.groupby(temp_cut)['불량여부'].mean() * 100

진행온도
(-0.137, 27.48]    15.582192
(27.48, 54.96]     15.994939
(54.96, 82.44]     15.712831
(82.44, 109.92]    14.531646
(109.92, 137.4]    14.661954
Name: 불량여부, dtype: float64

In [53]:
# 4) 상관분석 
dye[['지시온도', '진행온도', '염색색차_DE']].corr().round(2)     

,지시온도,진행온도,염색색차_DE
지시온도,1.00,0.99,0.01
진행온도,0.99,1.00,0.01
염색색차_DE,0.01,0.01,1.00


In [55]:
good=dye[dye['불량여부'] == 0]                  
good[['지시온도', '진행온도', '염색색차_DE']].corr().round(2)

,지시온도,진행온도,염색색차_DE
지시온도,1.00,0.99,-0.02
진행온도,0.99,1.00,-0.02
염색색차_DE,-0.02,-0.02,1.00


In [57]:
bad=dye[dye['불량여부'] == 1]
bad[['지시온도', '진행온도', '염색색차_DE']].corr().round(2)

# 결론] 지시온도, 진행온도, 공정시간은 분석(조건별, 구간분석, 상관분석)을 통해 원인 아님

,지시온도,진행온도,염색색차_DE
지시온도,1.00,0.99,0.1
진행온도,0.99,1.00,0.1
염색색차_DE,0.10,0.10,1.0


#### 원단
- 중량과 길이 : 투입중량/길이

In [60]:
# 1) 조건별 비교 : 투입중량/길이 평균값으로 원인 분석 
# 확연한 차이 (얇은 원단 또는 원단의 밀도 낮음)

weight=dye.groupby('불량여부')['투입중량/길이'].mean().round(2)
weight.reset_index()

,불량여부,투입중량/길이
0,0,5.51
1,1,1.50


In [62]:
# 2) 구간 분석 
# 저밀도 : 가벼움 / 같은 길이 무게가 적게 들어감 / 원단 조직 느슨함 / 실과 실사이 간격
# 고밀도 : 무거움 / 같은 길이 무게가 많이 들어감 / 원단 조직 치밀함 / 실과 실 사이 간격 좁음

w_cut=pd.cut(dye['투입중량/길이'], bins=10)           
dye.groupby(w_cut)['불량여부'].agg(['mean', 'sum', 'count'])

,mean,sum,count
투입중량/길이,,,
"(-0.0228, 14.105]",0.166493,4953,29749
"(14.105, 28.094]",0.000000,0,1740
"(28.094, 42.082]",0.000000,0,596
"(42.082, 56.07]",0.000000,0,283
"(126.012, 140.0]",0.000000,0,240


- cut : 값의 범위(Range) 등분, 모두 일정함
- qcut : 데이터 개수(Frequency)를 등분, 데이터 분포에 따라 유동적

In [65]:
w_qcut=pd.qcut(dye['투입중량/길이'], q=20, duplicates='drop')     # 10개의 구간으로 나눔, 중복값 없애기 
dye.groupby(w_qcut)['불량여부'].agg(['mean', 'count', 'sum'])

# 문제는 0.33 ~ 3.89 사이의 애매한 구간이며, 특히 2.74 ~ 3.89 사이에서는 불량률이 60%에 달한다.
# 추가적은 원단, 액량, 기계(설비, LOT), 포속 

,mean,count,sum
투입중량/길이,,,
"(0.116, 0.118]",0.000000,1643,0
"(0.118, 0.315]",0.000000,1876,0
"(0.315, 0.331]",0.000000,1520,0
"(0.331, 0.404]",0.315309,1535,484
"(0.404, 0.49]",0.141139,1580,223
"(0.49, 0.541]",0.094886,1760,167
"(0.541, 0.627]",0.417374,1531,639
"(0.627, 0.647]",0.344968,1719,593
"(0.647, 0.949]",0.085870,1840,158


In [67]:
# 3) 상관 분석 
bad=dye[dye['불량여부'] == 1]
bad[['투입중량/길이', '염색색차_DE']].corr().round(2)

,투입중량/길이,염색색차_DE
투입중량/길이,1.00,0.39
염색색차_DE,0.39,1.00


### 설비 

In [127]:
mach_no=dye.groupby('설비번호')

# 1) 설비별 불량률
mach_defect=mach_no['불량여부'].agg(['mean', 'sum', 'count'])     # 불량률, 불량건수, 전체건수 
mach_defect.sort_values(by='mean', ascending=False)

,mean,sum,count
설비번호,,,
FCM07,0.490102,817,1667
FCM05,0.309026,1907,6171
FCM12,0.266018,984,3699
FCM08,0.102770,920,8952
FCM11,0.073165,325,4442
FCM06,0.000000,0,5774
FCM15,0.000000,0,1903


In [242]:
# 1-1) 정상 설비 투입중량/길이 : 0.33 ~ 3.89 범위안에 안 들어간다.
good=dye['설비번호'].isin(['FCM06', 'FCM15'])    
dye[good].groupby('설비번호')['투입중량/길이'].agg(['count', 'mean'])

,count,mean
설비번호,,
FCM06,5774,6.221309
FCM15,1903,0.117954


In [244]:
# 1-2) 불량 설비 투입증량/길이    평균 : 0.33 ~ 3.89 범위안에 들어간다. 
fail=dye[dye['불량여부'] == 1]
fail.groupby('설비번호')['투입중량/길이'].mean().reset_index()       # 0.33 ~ 3.89 구간의 에러 

,설비번호,투입중량/길이
0,FCM05,1.044281
1,FCM07,0.637665
2,FCM08,2.175550
3,FCM11,0.692545
4,FCM12,2.717544


In [246]:
# 1-3) 정상 설비 투입증량/길이    평균 : FCM07 ??
success=dye[dye['불량여부'] == 0]
success.groupby('설비번호')['투입중량/길이'].mean().reset_index()     # 0.33 ~ 3.89 구간의 에러      

,설비번호,투입중량/길이
0,FCM05,7.219711
1,FCM06,6.221309
2,FCM07,0.653614
3,FCM08,4.190398
4,FCM11,9.400642
5,FCM12,4.590219
6,FCM15,0.117954


In [248]:
# 2) FCM07 구간분석 : 중간 얇기 문제 발생 >> LOT, 포속, 원단, 액상 데이터 확인 필요
bad=dye[dye['설비번호'] == 'FCM07']    # FCM05, FCM08, FCM11, FCM12
bad_cut=pd.cut(bad['투입중량/길이'], bins=5)

bad_analysis=bad.groupby(bad_cut)['불량여부'].agg(['count', 'sum', 'mean'])
bad_analysis.reset_index()

,투입중량/길이,count,sum,mean
0,"(0.578, 0.617]",211,0,0.000000
1,"(0.617, 0.656]",1247,817,0.655172
2,"(0.734, 0.773]",209,0,0.000000


- [결론]
- 공정(온도, 원단, 공정진행시간), 설비(설비번호) 불량률 원인분석(조건비교, 구간분석, 상관분석) 결과
- 투입중량/길이 : 0.33 ~ 3.89 불량 발생
- FCM06, FCM15  : 0.33 ~ 3.89 아닌 경우 불량률 0%
- FCM07 (불량률 49%) : 0.617 ~ 0.656 불량률 발생 이상 ?? 
    - 포속: 포속이 과도하게 빨라져 길이 대비 중량 밸런스가 깨지는가?
    - 원단: 특정 LOT나 원단 종류가 들어올 때 원단 밀도 문제로 수치가 이 구간에 갇히는가?
    - 액상: 액상 주입량이나 압력이 부족하여 수치가 위험 늪지대로 흘러내리는가?

## Pivot table
- 데이터프레임의 열을 재구조화하여 새로운 테이블 형태 만듬

In [409]:
dye.pivot_table(
    index='설비번호',              # 행으로 사용할 열 (그룹)
    values='염색색차_DE',          # 집계할 값 ['염색색차_DE', '지시온도']
    aggfunc='mean',                # 집계 함수 ['mean', 'max'],
    fill_value=0                   # 빈 값은 0으로 채움
)

# dye.groupby('설비번호')['염색색차_DE'].mean()

,염색색차_DE
설비번호,
FCM05,2.811437
FCM06,1.180983
FCM07,2.705375
FCM08,1.886812
FCM11,1.561275
FCM12,3.953758
FCM15,0.395969


## 병합
- merge() : 두 개의 데이터프레임을 특정 열(인덱스 또는 열)을 기준으로 결합
    - 형식 : pd.merge(left=df1, right=df2, left_on='column name', right_on='column name', how='join 방법')
    - left=df1, right=df2 : 병합할 두개의 데이터 프레임 지정
    - left_on='column name', right_on='column name': 병합 기준이 되는 열을 지정
    - how=’join 방법’ : 병합 방법 지정
        - inner join : 조인 조건에 일치하는 정보를 조회
        - left join : 왼쪽 정보를 기준으로 결합한다. (왼쪽 정보가 조건에 부합하지 않아도 모두 결합)
        - right join : 오른쪽 정보를 기준으로 결합한다. (오른쪽 정보가 조건에 부합하지 않아도 모두 결합)
        - outer join : 조인 조건에 일치하지 않는 정보까지 모두 조회

In [412]:
df1=pd.DataFrame([[101, 'A'], 
                  [201, 'B'],
                  [301, 'C']],
                  columns=['id', 'value_df1'])
df1

,id,value_df1
0,101,A
1,201,B
2,301,C


In [414]:
df2=pd.DataFrame({'id':[201, 301, 401], 
                  'value_df2':['X', 'Y', 'Z']})
df2

,id,value_df2
0,201,X
1,301,Y
2,401,Z


In [416]:
# inner join : 조인 조건에 일치하는 정보를 조회
result_inner=pd.merge(left=df1, right=df2, left_on='id', right_on='id', how='inner')
result_inner      

# 조인 조건 : 병합할 두개의 데이터 프레임 지정, 병합 기준이 되는 열을 지정, 조인 조건에 일치하는 정보를 조회

,id,value_df1,value_df2
0,201,B,X
1,301,C,Y


In [418]:
# left join : 왼쪽 정보를 기준으로 결합한다. (왼쪽 정보가 조건에 부합하지 않아도 모두 결합)

result_left=pd.merge(left=df1, right=df2, left_on='id', right_on='id', how='left')
result_left

,id,value_df1,value_df2
0,101,A,NaN
1,201,B,X
2,301,C,Y


In [420]:
# right join : 오른쪽 정보를 기준으로 결합한다. (오른쪽 정보가 조건에 부합하지 않아도 모두 결합)

result_right=pd.merge(left=df1, right=df2, left_on='id', right_on='id', how='right')
result_right

,id,value_df1,value_df2
0,201,B,X
1,301,C,Y
2,401,NaN,Z


In [422]:
# outer join : 조인 조건에 일치하지 않는 정보까지 모두 조회

result_outer=pd.merge(left=df1, right=df2, left_on='id', right_on='id', how='outer')
result_outer

,id,value_df1,value_df2
0,101,A,NaN
1,201,B,X
2,301,C,Y
3,401,NaN,Z


In [ ]:
# End of -------------------------------------------------